# E21 · 4-State Single-Prefill Router (S4R)
## 해결책 — 박세령 4-state 판독을 주 방법으로 연결하는 행동 라우터
**주 경로:** 질문+문맥 하나 → L13 역할별 선택 헤드의 마지막 벡터 → 4-state LR → retain / accept / no-op → L14–18 같은 prefill에서 개입 → 하나의 답변.

Packed, A/B 점수, 추가 closed-book 분기, 후보 답변 비교를 **온라인 경로에서 사용하지 않습니다.**
이 노트북은 E16 원형 `role top-4 → union → HEAD_LAST → LR`을 새로 학습합니다. E17의 `HQ + (HL−HQ) + C_MASS` 판독기로 몰래 바꾸지 않습니다.

**실행 상태:** 배포 시점 Llama-3.1-8B GPU 추론/실험 성능은 미실행. CPU 단위검증과 테스트 대역의 hook/생성 경로 검증은 별도 `VALIDATION.json`을 참고합니다. 실제 성능 숫자는 실행 후에만 생성됩니다.

**실행:** 새 Colab A100 40GB급 런타임 → 위에서부터 실행 → 기존 자료에 접근 가능한 Google 계정 연결 → HF_TOKEN 인증. 필요한 원본 4개는 확인된 Drive ID로 자동 확보합니다. 기존 E14/E17/E19 산출물을 업로드할 필요가 없습니다. 원본 노트북/폴더를 수정하지 않습니다.

**분할:** E14/E17 fold-0의 fit 366 / utility_fit 61 / policy_cal 62 / test 123 질문. 판독기는 fit만 사용합니다. 행동 라우터는 fit의 5-fold OOF 판독확률과 utility_fit의 미학습 질문 판독확률을 사용합니다. Head 선발·표준화·LR을 OOF마다 재학습합니다. **소량 라벨 20% 재현 또는 5-fold 최종 QA 평가라고 부르지 않습니다.**

**기존 결과를 본 후 개발하는 탐색적 후속 실험입니다.** 현재 실행에서 test 재보정을 금지하지만, 이미 여러 차례 사용한 p8s/NQ/ConFiQA를 새 blind test라고 주장하지 않습니다.

In [ ]:
# 01. 환경 및 고정 설정 — 설치 전에 GPU 확인
import os,sys,subprocess,json,hashlib,time,inspect,types,zipfile
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 A100급 GPU 런타임을 선택하세요. 이 셀은 8B 모델을 CPU에 로드하지 않습니다.')
free,total=torch.cuda.mem_get_info()
if total/2**30 < 35 or not torch.cuda.is_bf16_supported():
    raise RuntimeError('이 full BF16 프로토콜은 A100 40GB급 이상을 기준으로 합니다. 양자화로 자동 변경하지 않습니다.')
if 'transformers' in sys.modules and sys.modules['transformers'].__version__!='4.57.1':
    raise RuntimeError('런타임을 재시작하고 첫 셀부터 실행하세요.')
if 'sklearn' in sys.modules and sys.modules['sklearn'].__version__!='1.8.0':
    raise RuntimeError('scikit-learn 버전 충돌: 새 런타임이 필요합니다.')
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1',
                      'scikit-learn==1.8.0','accelerate','safetensors','pyarrow',
                      'google-api-python-client','huggingface_hub','tabulate'])
CFG={
 'RUN_TAG':'E21_S4R_20261104_v1','CODE_VERSION':'S4R-1.0.0','SEED':42,'FOLD':0,
 'MODEL':'meta-llama/Llama-3.1-8B-Instruct',
 'REVISION':'0e9e39f249a16976918f6564b8830bc894c89659',
 'READ_LAYER':13,'WRITE_LAYERS':[14,15,16,17,18],
 'ROLE_TOPK':4,'READOUT_C':.1,'ROUTER_C':.1,'OOF_FOLDS':5,
 'ALPHA':.5,'DELTA_CAP':.05,'DIRECTION_QUESTIONS':64,
 'BATCH_SIZE':2,'MAX_PROMPT_TOKENS':4096,'MAX_NEW_TOKENS':192,
 'HARM_WEIGHT':2.,'MIN_TRAIN_GAINS':3,
 'MARGIN_GRID':[0.,.01,.025,.05,.1,.2],'HARM_GRID':[.01,.025,.05,.1,1.],
 'NQ_QUESTIONS':1000,'CONFIQA_QUESTIONS':1000,'BOOTSTRAP':2000,
 'BENCHMARK_PROMPTS':12,'BENCHMARK_REPEATS':3,
 'OUTPUT_PARENT':'/content/drive/MyDrive/p8s',
 'PRIMARY_METHOD':'S4R_gain_harm',
}
print('GPU:',torch.cuda.get_device_name(0),'| 실험:',CFG['RUN_TAG'])

## 02–05. CPU 판독·라우터·평가 코드
기존 판독의 수식/상태 순서 `[K+C+, K+C−, K−C−, K−C+]`를 유지합니다.
새 행동 라우터는 각 개입의 **회복(Gain)**, **손상(Harm)**, **정오 변화 없음**을 학습합니다. 실제 입력은 판독된 4-state 확률뿐입니다. `G − 2H`는 이번 실험에서 고정한 효용 점수이며, 통계적으로 보증된 확률/안전성으로 해석하지 않습니다.

In [ ]:
"""E21 S4R: original role-head 4-state readout and outcome-aware action routing.
No LLM weights or benchmark results are included. Runtime routing receives p(4-state) only.
"""
from __future__ import annotations
import ast
import hashlib
import itertools
import json
import re
import string
import unicodedata
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

STATES = ('K+C+', 'K+C-', 'K-C-', 'K-C+')
ACTIONS = ('noop', 'retain', 'accept')

In [ ]:
# Serialization, scoring, and source-compatible split

def jsonable(x: Any) -> Any:
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, np.generic): return x.item()
    if isinstance(x, Path): return str(x)
    if isinstance(x, dict): return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (tuple, list)): return [jsonable(v) for v in x]
    return x

def canonical(x: Any) -> str:
    return json.dumps(jsonable(x), ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False)

def digest(x: Any) -> str:
    return hashlib.sha256(canonical(x).encode()).hexdigest()

def file_sha(path) -> str:
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def atomic_json(path, value) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp')
    tmp.write_text(canonical(value), encoding='utf-8'); tmp.replace(path)

def lock_json(path, value) -> None:
    path = Path(path)
    if path.exists() and json.loads(path.read_text()) != jsonable(value):
        raise ValueError(f'Immutable manifest changed: {path}. Use a new RUN_TAG.')
    if not path.exists(): atomic_json(path, value)

def question_key(x: str) -> str:
    return ' '.join(unicodedata.normalize('NFKC', str(x)).casefold().split())

def norm_answer(x: str) -> str:
    s = unicodedata.normalize('NFKC', str(x)).lower()
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(w for w in s.split() if w not in {'a','an','the'})

def answer_extract(x: str) -> str:
    s = str(x).strip().split('\n')[0].strip()
    s = re.sub(r'^(answer|the answer is)\s*:?\s*', '', s, flags=re.I)
    return re.sub(r'[.,;:\s]+$', '', s.split('. ')[0].strip().strip('"\''))

def answer_em(raw, aliases) -> int:
    return int(norm_answer(answer_extract(raw)) in {norm_answer(x) for x in aliases if norm_answer(x)})

def raw_em(raw, aliases) -> int:
    return int(norm_answer(raw) in {norm_answer(x) for x in aliases if norm_answer(x)})

def alias_list(x) -> list[str]:
    if x is None: return []
    if isinstance(x, np.ndarray): x = x.tolist()
    if isinstance(x, (list, tuple)):
        return list(dict.fromkeys(a for v in x for a in alias_list(v)))
    if not isinstance(x, str) and bool(pd.isna(x)): return []
    # Preserve actual strings 'Nan' and 'NA'.
    return [str(x).strip()] if str(x).strip() else []

def parse_gold(x) -> list[str]:
    if isinstance(x, str):
        try: x = json.loads(x)
        except (ValueError, TypeError):
            try: x = ast.literal_eval(x)
            except (ValueError, SyntaxError): pass
    return [v for v in alias_list(x) if norm_answer(v)]

def state_ids(k, c) -> np.ndarray:
    k,c = np.asarray(k,int), np.asarray(c,int)
    if not np.isin(k,[0,1]).all() or not np.isin(c,[0,1]).all():
        raise ValueError('K and C must be binary.')
    return np.where(k==1, np.where(c==1,0,1), np.where(c==1,3,2)).astype(int)

def load_p8s(behavior, klabels) -> pd.DataFrame:
    f = pd.read_csv(behavior, keep_default_na=False).reset_index(drop=True)
    lab = pd.read_parquet(klabels)
    lab['k'] = lab.kside_corr.map({'K+':1,'K-':0,'K−':0})
    if lab.k.isna().any() or not lab.groupby('question').k.nunique().eq(1).all():
        raise ValueError('Missing or conflicting corrected K labels.')
    km = lab.drop_duplicates('question').set_index('question').k
    f['k'] = f.question.map(km)
    f['c'] = f.ctx_label.map({'C+':1,'C-':0,'C−':0})
    if f[['k','c']].isna().any().any(): raise ValueError('Unmatched p8s K/C.')
    f[['k','c']] = f[['k','c']].astype(int)
    f['state'] = state_ids(f.k, f.c); f['row'] = np.arange(len(f))
    f['aliases'] = f.gold.map(parse_gold)
    if len(f)!=1224 or f.question.nunique()!=612: raise ValueError('Expected 612 paired questions.')
    if not f.groupby('question').size().eq(2).all() or not f.groupby('question').c.nunique().eq(2).all():
        raise ValueError('Incomplete p8s C± pairs.')
    if f.groupby('state').size().reindex(range(4)).tolist()!=[362,362,250,250]:
        raise ValueError('Corrected K distribution differs from the source E17 loader.')
    if f.question.map(question_key).nunique()!=612: raise ValueError('Normalized p8s question collision.')
    return f

def e14_split(frame, fold=0, seed=42) -> pd.DataFrame:
    if fold not in range(5): raise ValueError('fold must be 0..4.')
    q = frame[['question','k']].drop_duplicates().sort_values('question')
    rng=np.random.default_rng(seed); mapping={}
    for cls in (0,1):
        ids=q.loc[q.k.eq(cls),'question'].to_numpy().copy(); rng.shuffle(ids)
        mapping.update({s:i%5 for i,s in enumerate(ids)})
    folds=frame.question.map(mapping).to_numpy()
    roles=np.full(len(frame),'fit',object)
    roles[folds==fold]='test'; roles[folds==(fold+1)%5]='cal'
    rng=np.random.default_rng(seed+1000*fold)
    for cls in (0,1):
        qs=np.array(sorted(set(frame.loc[(roles=='cal') & frame.k.eq(cls),'question'])))
        rng.shuffle(qs); half=len(qs)//2
        roles[(roles=='cal') & frame.question.isin(qs[:half])]='utility_fit'
        roles[(roles=='cal') & frame.question.isin(qs[half:])]='policy_cal'
    out=frame[['row','question','k','c','state']].copy()
    out['outer_fold']=folds; out['role']=roles
    if not out.groupby('question').role.nunique().eq(1).all(): raise ValueError('Split leakage.')
    return out

In [ ]:
# Original E16 role top-4 -> HEAD_LAST -> LR

def role_heads(hq, hl, cm, states, topk=4):
    hq,hl,cm=np.asarray(hq,np.float32),np.asarray(hl,np.float32),np.asarray(cm,np.float32)
    if hq.shape!=hl.shape or hq.ndim!=3 or cm.shape!=hq.shape[:2]: raise ValueError('Head bank shape mismatch.')
    if not all(np.isfinite(x).all() for x in (hq,hl,cm)): raise ValueError('Non-finite head bank.')
    s=np.asarray(states,int); k=np.isin(s,[0,1]); c=np.isin(s,[0,3])
    n=np.linalg.norm(hq,axis=-1)
    scores={'memory':n,'context':cm,'conflict':np.linalg.norm(hl-hq,axis=-1)/np.maximum(n,1e-8)}
    targets={'memory':k,'context':c,'conflict':k!=c}; roles={}; ranking={}
    for name,z in scores.items():
        if np.unique(targets[name]).size!=2: raise ValueError('Both role labels required for head selection.')
        auc=[float(roc_auc_score(targets[name],z[:,h])) for h in range(z.shape[1])]
        order=sorted(range(z.shape[1]),key=lambda h:(-max(auc[h],1-auc[h]),h))
        roles[name]=order[:min(topk,len(order))]; ranking[name]=auc
    return sorted(set(h for hs in roles.values() for h in hs)),roles,ranking

def fit_linear(x,y,C=.1,seed=42,max_iter=4000):
    x=np.asarray(x,float); y=np.asarray(y,int)
    if x.ndim!=2 or len(x)!=len(y) or not np.isfinite(x).all(): raise ValueError('Invalid LR data.')
    classes=np.unique(y)
    if len(classes)==1:
        return {'kind':'constant','classes':classes.tolist(),'n_features':x.shape[1]}
    scaler=StandardScaler().fit(x)
    model=LogisticRegression(C=C,solver='lbfgs',max_iter=max_iter,random_state=seed,tol=1e-7)
    with warnings.catch_warnings(),threadpool_limits(limits=1):
        warnings.simplefilter('error',ConvergenceWarning)
        model.fit(scaler.transform(x),y)
    return {'kind':'logistic','classes':model.classes_.tolist(),'mean':scaler.mean_.tolist(),
            'scale':scaler.scale_.tolist(),'coef':model.coef_.tolist(),'intercept':model.intercept_.tolist(),
            'n_features':x.shape[1]}

def predict_linear(model,x):
    x=np.asarray(x,float)
    if x.ndim!=2 or x.shape[1]!=model['n_features'] or not np.isfinite(x).all():
        raise ValueError('Frozen classifier feature contract violated.')
    if model['kind']=='constant': return np.ones((len(x),1))
    z=((x-np.asarray(model['mean']))/np.asarray(model['scale'])) @ np.asarray(model['coef']).T + np.asarray(model['intercept'])
    if len(model['classes'])==2:
        v=z[:,0]; p=np.empty(len(v)); positive=v>=0
        p[positive]=1/(1+np.exp(-v[positive])); ev=np.exp(v[~positive]);p[~positive]=ev/(1+ev)
        return np.column_stack([1-p,p])
    z=z-z.max(axis=1,keepdims=True); p=np.exp(z)
    return p/p.sum(axis=1,keepdims=True)

def fit_readout(hq,hl,cm,y,topk=4,C=.1,seed=42):
    heads,roles,ranking=role_heads(hq,hl,cm,y,topk)
    x=np.asarray(hl,np.float32)[:,heads,:].reshape(len(hl),-1)
    model=fit_linear(x,y,C,seed)
    if model['classes']!=[0,1,2,3]: raise ValueError('All four state classes required.')
    return {'schema':'E16_role_top4_HEAD_LAST_LR','heads':heads,'roles':roles,'role_auc':ranking,
            'model':model,'runtime_features':'selected HEAD_LAST only', 'label_budget_rows':len(y)}

def read_probs(reader,hl):
    a=np.asarray(hl,np.float32)
    return predict_linear(reader['model'],a[:,reader['heads'],:].reshape(len(a),-1))

def oof_readout(hq,hl,cm,states,questions,nfold=5,seed=42,topk=4,C=.1):
    """Reselect heads, standardize and refit LR inside every held-out question fold."""
    q=np.asarray(questions,str);y=np.asarray(states,int);uq=np.unique(q)
    k=np.array([int(y[q==v][0] in (0,1)) for v in uq])
    if np.bincount(k,minlength=2).min()<nfold: raise ValueError('Insufficient question classes for OOF.')
    probs=np.full((len(y),4),np.nan);seen=np.zeros(len(y),int);audit=[]
    for fold,(tr,va) in enumerate(StratifiedKFold(n_splits=nfold,shuffle=True,random_state=seed).split(uq,k)):
        train=np.flatnonzero(np.isin(q,uq[tr]));valid=np.flatnonzero(np.isin(q,uq[va]))
        assert not set(q[train]) & set(q[valid])
        r=fit_readout(hq[train],hl[train],cm[train],y[train],topk,C,seed+fold)
        probs[valid]=read_probs(r,hl[valid]);seen[valid]+=1
        audit.append({'fold':fold,'train_questions':uq[tr].tolist(),'heldout_questions':uq[va].tolist(),
                      'heads':r['heads'],'roles':r['roles'],'heldout_rows':valid.tolist()})
    if not (seen==1).all() or not np.isfinite(probs).all(): raise AssertionError('Incomplete OOF.')
    return probs,audit

In [ ]:
# Gain/harm routing and state-information ablations

def check_probs(p):
    p=np.asarray(p,float)
    if p.ndim!=2 or p.shape[1]!=4 or not np.isfinite(p).all() or (p<0).any() or not np.allclose(p.sum(1),1):
        raise ValueError('Expected four state probabilities in the frozen class order.')
    return p

def routing_features(p,mode='P4'):
    p=check_probs(p)
    if mode=='P4': return p
    if mode=='K_only': return (p[:,0]+p[:,1])[:,None]
    if mode=='C_only': return (p[:,0]+p[:,3])[:,None]
    if mode=='Conflict_only': return (p[:,1]+p[:,3])[:,None]
    raise ValueError('Unknown routing feature mode.')

def fit_outcome_router(p,em,mode='P4',C=.1,harm_weight=2.,min_gains=3,seed=42):
    """Train on OFFLINE forced-action results; features are only predicted state signals.
    Transition labels: 0 unchanged EM, 1 baseline-wrong -> correct, 2 correct -> wrong.
    """
    p=check_probs(p);e=np.asarray(em,int)
    if e.shape!=(len(p),3) or not np.isin(e,[0,1]).all(): raise ValueError('Expected [noop,retain,accept] EM.')
    x=routing_features(p,mode);models={}
    for action in (1,2):
        target=np.where((e[:,0]==0)&(e[:,action]==1),1,np.where((e[:,0]==1)&(e[:,action]==0),2,0))
        counts=np.bincount(target,minlength=3)
        models[str(action)]={'model':fit_linear(x,target,C,seed),
            'counts':counts.tolist(),'prior':((counts+1)/(len(target)+3)).tolist(),
            'shrinkage':float(3/(len(target)+3)),
            'enabled':bool(counts[1]>=min_gains)}
    return {'kind':'gain_harm','feature_mode':mode,'models':models,'harm_weight':float(harm_weight),
            'min_observed_gains':int(min_gains),'training_rows':len(p),
            'features':'state probabilities only; no candidate, label, first-token or answer features'}

def outcome_scores(router,p):
    p=check_probs(p);x=routing_features(p,router['feature_mode']);out=np.zeros((len(p),3,3))
    out[:,0,0]=1
    for a in (1,2):
        m=router['models'][str(a)];base=predict_linear(m['model'],x);mapped=np.zeros((len(x),3))
        mapped[:,m['model']['classes']]=base
        out[:,a,:]=(1-m['shrinkage'])*mapped + m['shrinkage']*np.asarray(m['prior'])
    return out

def choose_actions(router,p,guard):
    p=check_probs(p)
    if guard.get('force_noop',False): return np.zeros(len(p),int)
    if router['kind']=='rule':
        s=p.argmax(1);a=np.where(s==1,1,np.where(s==3,2,0))
        a[p.max(1)<float(guard.get('confidence',0.))]=0
        return a
    if router['kind']=='fixed': return np.full(len(p),int(router['action']),int)
    score=outcome_scores(router,p)
    util=score[:,:,1]-router['harm_weight']*score[:,:,2];util[:,0]=0.
    for a in (1,2):
        eligible=(score[:,a,2]<=guard['max_harm']) & (util[:,a]>guard['margin'])
        if not router['models'][str(a)]['enabled']: eligible[:]=False
        util[:,a]=np.where(eligible,util[:,a],-np.inf)
    return util.argmax(1).astype(int)

def preservation(base,new,mask=None):
    b=np.asarray(base,bool);n=np.asarray(new,bool)
    mask=np.ones(len(b),bool) if mask is None else np.asarray(mask,bool)
    b,n=b[mask],n[mask];kept=int((b&n).sum());lost=int((b&~n).sum());gain=int((~b&n).sum())
    return {'n':len(b),'base_correct':int(b.sum()),'correct':int(n.sum()),'kept':kept,
            'lost':lost,'gained':gain,'net':gain-lost,
            'EM':float(n.mean()) if len(n) else None,
            'preservation':kept/int(b.sum()) if b.sum() else None}

def calibrate_policy(router,p,em,cplus,known,margin_grid=(0.,.01,.025,.05,.1,.2),
                     harm_grid=(.01,.025,.05,.1,1.),min_cal_questions=30,questions=None):
    """Empirical dev preservation, NOT a statistical guarantee. Always retain full grid."""
    p=check_probs(p);e=np.asarray(em,int);cp=np.asarray(cplus,bool);kmcp=cp&~np.asarray(known,bool)
    if e.shape!=(len(p),3): raise ValueError('Cal EM shape mismatch.')
    adequate=(len(set(questions))>=min_cal_questions if questions is not None else False)
    adequate &= int((cp&e[:,0].astype(bool)).sum())>=20 and int((kmcp&e[:,0].astype(bool)).sum())>=5
    guards=[{'force_noop':True,'margin':1.01,'max_harm':0.,'confidence':1.01}]
    if router['kind']=='rule':
        guards += [{'force_noop':False,'confidence':v} for v in (0.,.6,.7,.8,.9,.95)]
    elif router['kind']=='gain_harm':
        guards += [{'force_noop':False,'margin':m,'max_harm':h} for m,h in itertools.product(margin_grid,harm_grid)]
    else: raise ValueError('Calibrator supports rule and gain_harm routers.')
    rows=[]
    for i,g in enumerate(guards):
        a=choose_actions(router,p,g);new=e[np.arange(len(e)),a]
        total=preservation(e[:,0],new);cpp=preservation(e[:,0],new,cp);kmp=preservation(e[:,0],new,kmcp)
        feasible=(i==0 or (adequate and cpp['lost']==0 and kmp['lost']==0 and total['net']>0))
        rows.append({'candidate':i,**g,**total,'Cplus_lost':cpp['lost'],'KminusCplus_lost':kmp['lost'],
                     'switches':int((a!=0).sum()),'support_ok':bool(adequate),'eligible':bool(feasible)})
    good=[r for r in rows if r['eligible']]
    winner=min(good,key=lambda r:(-r['correct'],r['lost'],r['switches'],r['candidate']))
    idx=winner['candidate']; guard=dict(guards[idx])
    guard['reason']='dev_noop_fallback' if idx==0 else 'dev_preserved_and_improved'
    guard['selected_on']='p8s policy_cal only'
    for r in rows:r['selected']=r['candidate']==idx
    return guard,pd.DataFrame(rows)

In [ ]:
# Paired clustered reporting

def cluster_delta_ci(base,new,questions,mask=None,n_boot=2000,seed=42):
    b=np.asarray(base,float);n=np.asarray(new,float);q=np.asarray([question_key(x) for x in questions])
    mask=np.ones(len(b),bool) if mask is None else np.asarray(mask,bool)
    if not mask.any(): return None,None,None
    d=pd.DataFrame({'q':q[mask],'d':(n-b)[mask]}).groupby('q').d.agg(['sum','count'])
    sums=d['sum'].to_numpy();counts=d['count'].to_numpy();rng=np.random.default_rng(seed);est=[]
    for _ in range(n_boot):
        ix=rng.integers(0,len(d),len(d));est.append(sums[ix].sum()/counts[ix].sum())
    lo,hi=np.quantile(est,[.025,.975]);return float((n-b)[mask].mean()),float(lo),float(hi)

def evaluate_outputs(meta,outputs,known,n_boot=2000,seed=42):
    """All labels join AFTER online generation. Current standalone-CB K is evaluation-only."""
    y=state_ids(known,np.asarray(meta.c,int));base=outputs['Base'];rows=[];changes=[]
    base_em=np.array([answer_em(r['raw'],g) for r,g in zip(base,meta.aliases)])
    masks={'all':np.ones(len(meta),bool),'C+':meta.c.to_numpy()==1,'C-':meta.c.to_numpy()==0,
           **{s:y==i for i,s in enumerate(STATES)}}
    for name,recs in outputs.items():
        e=np.array([answer_em(r['raw'],g) for r,g in zip(recs,meta.aliases)])
        raw_e=np.array([raw_em(r['raw'],g) for r,g in zip(recs,meta.aliases)])
        for subset,mask in masks.items():
            pr=preservation(base_em,e,mask);delta,lo,hi=cluster_delta_ci(base_em,e,meta.question,mask,n_boot,seed)
            rows.append({'method':name,'subset':subset,**pr,'delta_EM':delta,'CI_low':lo,'CI_high':hi,
                         'raw_EM':float(raw_e[mask].mean()) if mask.any() else None,
                         'intervention_rate':float(np.mean([r['action']!='noop' for r,m in zip(recs,mask) if m])) if mask.any() else None,
                         'n_original_ids':int(meta.loc[mask,'qid'].nunique()),
                         'n_question_clusters':int(meta.loc[mask,'question'].map(question_key).nunique())})
        for i in np.flatnonzero(e!=base_em):
            changes.append({'method':name,'qid':str(meta.iloc[i].qid),'variant':meta.iloc[i].variant,
                            'question':meta.iloc[i].question,'state':STATES[y[i]],
                            'change':'gained' if e[i] else 'lost','Base':base[i]['raw'],'new':recs[i]['raw'],
                            'action':recs[i]['action'],'pstate':recs[i].get('pstate')})
    return pd.DataFrame(rows),pd.DataFrame(changes),base_em,y

## 06–09. 실제 single-prefill 엔진
온라인 판독은 `HEAD_LAST`만 읽습니다. HQ·C_MASS는 **오프라인 헤드 선별용**으로만 계산합니다.
L13에서 결정하고 L14–18에서 개입하므로 프롬프트를 다시 실행하지 않습니다. decode에는 개입을 반복하지 않습니다. ordinary batch의 여러 질문은 서로 별개의 입력이며, 한 질문의 A/B branch가 아닙니다.
기존 E17의 prefill+decode 스케줄과 다르므로 E19처럼 **prefill-only 대조군끼리** 비교합니다.

In [ ]:
"""Single-context, single-prefill runtime. No A/B streams and no answer-bank input.
The Llama 8B run is performed in Colab; this module also supports CPU test fixtures.
"""
from __future__ import annotations
import time
from dataclasses import dataclass
import numpy as np
import torch

SYSTEM_PROMPT = ('You are a question answering system. Answer the question based on the given context. '
    'Output the answer string itself and nothing else: no sentence, no explanation, '
    'no punctuation, no quotes. Keep it as short as possible (a name or a term).')
CB_SYSTEM_PROMPT = ('You are a question answering system. '
    'Output the answer string itself and nothing else: no sentence, no explanation, '
    'no punctuation, no quotes. Keep it as short as possible (a name or a term).')

@dataclass(frozen=True)
class PromptInput:
    # Deliberately excludes gold, qid, variant, K/C labels and previously generated answers.
    question: str
    context: str | None

In [ ]:
# Input preparation — ordinary left-padded batches, not A/B packing

class LLMEngine:
    def __init__(self,model,tokenizer,read_layer=13,write_layers=(14,15,16,17,18),
                 max_prompt_tokens=4096,max_new_tokens=192,alpha=.5,cap=.05):
        self.model=model.eval();self.tok=tokenizer
        self.read_layer=int(read_layer);self.write_layers=tuple(write_layers)
        if not self.write_layers or min(self.write_layers)<=self.read_layer: raise ValueError('Writes must follow read layer.')
        if not 0 <= self.read_layer < len(model.model.layers) or max(self.write_layers)>=len(model.model.layers):
            raise ValueError('Read/write layer index is out of bounds.')
        if len(set(self.write_layers))!=len(self.write_layers):raise ValueError('Duplicate write layers.')
        self.device=next(model.parameters()).device
        self.nheads=model.config.num_attention_heads
        self.nkv=model.config.num_key_value_heads
        self.hdim=model.config.hidden_size//self.nheads
        self.max_prompt_tokens=int(max_prompt_tokens);self.max_new_tokens=int(max_new_tokens)
        self.alpha=float(alpha);self.cap=float(cap)
        if self.max_new_tokens<1 or self.max_prompt_tokens<1 or not np.isfinite([self.alpha,self.cap]).all() or self.alpha<0 or self.cap<0:
            raise ValueError('Invalid generation length or intervention scale.')
        self.eos={int(tokenizer.eos_token_id)}
        eot=tokenizer.convert_tokens_to_ids('<|eot_id|>')
        if isinstance(eot,int) and eot>=0 and eot!=getattr(tokenizer,'unk_token_id',None): self.eos.add(eot)
    def sync(self):
        if self.device.type=='cuda': torch.cuda.synchronize(self.device)
    def render(self,item,directive=None):
        if type(item) is not PromptInput: raise TypeError('Only label-free PromptInput is accepted.')
        if not item.question.strip(): raise ValueError('Empty question.')
        cb=item.context is None
        if cb and directive is not None: raise ValueError('No instruction contrast in the CB evaluator.')
        system=CB_SYSTEM_PROMPT if cb else SYSTEM_PROMPT
        if directive=='context':system+=' For this example, rely on the context rather than remembered information.'
        elif directive=='memory':system+=' For this example, rely on your own knowledge and ignore the context.'
        elif directive is not None:raise ValueError('Invalid directive.')
        user='Question: '+item.question+('' if cb else '\nContext: '+item.context)+'\nAnswer:'
        txt=self.tok.apply_chat_template([{'role':'system','content':system},{'role':'user','content':user}],
                                         tokenize=False,add_generation_prompt=True)
        base=txt.rfind(user)
        if base<0:raise ValueError('Prompt span missing.')
        enc=self.tok(txt,add_special_tokens=False,return_offsets_mapping=True)
        qs=base+len('Question: ');qe=qs+len(item.question)
        q=[i for i,(s,e) in enumerate(enc['offset_mapping']) if e>qs and s<qe and e>s]
        c=[]
        if not cb:
            cs=qe+len('\nContext: ');ce=cs+len(item.context)
            c=[i for i,(s,e) in enumerate(enc['offset_mapping']) if e>cs and s<ce and e>s]
            if not c or not q or max(q)>=min(c):raise ValueError('Q before C invariant failed.')
        if not q or len(enc['input_ids'])>self.max_prompt_tokens:raise ValueError('Invalid/overlong input; no silent truncation.')
        return {'ids':enc['input_ids'],'q':q[-1],'c':c}
    def batch(self,inputs,directive=None):
        if not inputs:raise ValueError('Empty batch.')
        entries=[self.render(x,directive) for x in inputs];T=max(len(x['ids']) for x in entries);B=len(entries)
        ids=torch.full((B,T),int(self.tok.pad_token_id),dtype=torch.long,device=self.device)
        mask=torch.zeros_like(ids);qidx=[];cidx=[]
        for j,e in enumerate(entries):
            off=T-len(e['ids']);ids[j,off:]=torch.as_tensor(e['ids'],device=self.device);mask[j,off:]=1
            qidx.append(off+e['q']);cidx.append([off+i for i in e['c']])
        pos=(mask.cumsum(-1)-1).clamp_min(0)
        return {'ids':ids,'mask':mask,'pos':pos,'qidx':torch.tensor(qidx,device=self.device),
                'cidx':cidx,'tokens':[len(e['ids']) for e in entries]}
    @torch.inference_mode()
    def generate(self,inputs,reader=None,policy=None,vectors=None,force_action=0,capture=False):
        if (reader is None)!=(policy is None):raise ValueError('State reader and policy must be provided together.')
        if reader is not None and force_action!=0:raise ValueError('A live router cannot also force an action.')
        if any(x.context is None for x in inputs) and (reader is not None or capture or force_action!=0):
            raise ValueError('Closed-book is an isolated offline evaluator, never a router branch.')
        pkt=self.batch(inputs);B,T=pkt['ids'].shape
        session=RoutingSession(self,pkt,reader,policy,vectors,force_action,capture)
        self.sync()
        if self.device.type=='cuda':torch.cuda.reset_peak_memory_stats(self.device)
        started=time.perf_counter();parts=[];ended=torch.zeros(B,dtype=torch.bool,device=self.device)
        lengths=torch.zeros(B,dtype=torch.long,device=self.device);mask=pkt['mask']
        eos_t=torch.tensor(sorted(self.eos),device=self.device)
        with session:
            out=self.model.model(input_ids=pkt['ids'],attention_mask=mask,position_ids=pkt['pos'],
                                 cache_position=torch.arange(T,device=self.device),use_cache=True,return_dict=True)
            cache=out.past_key_values;logits=self.model.lm_head(out.last_hidden_state[:,-1,:])
            self.sync();ttft=time.perf_counter()-started
            for step in range(self.max_new_tokens):
                token=logits.argmax(-1)
                token=torch.where(ended,torch.full_like(token,int(self.tok.pad_token_id)),token)
                active=~ended;lengths+=active.long();parts.append(token.detach().cpu())
                ended|=torch.isin(token,eos_t)&active
                if bool(ended.all()) or step+1==self.max_new_tokens:break
                mask=torch.cat([mask,torch.ones((B,1),dtype=mask.dtype,device=self.device)],dim=-1)
                out=self.model.model(input_ids=token[:,None],attention_mask=mask,
                    position_ids=(mask.sum(-1)-1)[:,None],cache_position=torch.tensor([mask.shape[1]-1],device=self.device),
                    past_key_values=cache,use_cache=True,return_dict=True)
                cache=out.past_key_values;logits=self.model.lm_head(out.last_hidden_state[:,-1,:])
        self.sync();elapsed=time.perf_counter()-started
        peak=int(torch.cuda.max_memory_allocated(self.device)) if self.device.type=='cuda' else None
        if session.prefill_calls!=1 or (reader is not None and session.route_calls!=1):
            raise AssertionError('Single-prefill/router invariant failed.')
        if session.decode_writes!=0:raise AssertionError('Unexpected decode intervention.')
        if session.proposed_max>self.cap+1e-6:raise AssertionError('Proposed residual cap exceeded.')
        tokens=torch.stack(parts,1).numpy();records=[]
        for j,t in enumerate(tokens):
            stop=next((i for i,v in enumerate(t) if int(v) in self.eos),len(t))
            seq=t[:stop].tolist()
            records.append({'raw':self.tok.decode(seq,skip_special_tokens=True),'token_ids':seq,
                'action':ACTIONS[int(session.actions[j])],
                'pstate':None if session.probs is None else session.probs[j].tolist(),
                'pred_state':None if session.probs is None else int(session.probs[j].argmax()),
                'generated_tokens':len(seq),'prefill_calls':session.prefill_calls,'route_calls':session.route_calls,
                'decode_forward_calls':session.forward_calls-1,'decode_writes':session.decode_writes,
                'input_streams':1,'generation_trajectories':1,'online_closedbook_calls':0,
                'prompt_tokens':int(pkt['tokens'][j]),'padded_prompt_tokens':int(T),
                'proposed_delta_max':float(session.proposed_ratios[j]),
                'effective_delta_max':float(session.actual_ratios[j]),
                'batch_seconds':elapsed,'batch_size':B,'seconds_per_row':elapsed/B,
                'batch_ttft_seconds':ttft,'peak_allocated_bytes':peak,
                'trace':session.trace,'timing_scope':'model+router+decode; excludes tokenization and text decoding'})
        features=None
        if capture:
            features={'HEAD_Q':session.hq,'HEAD_LAST':session.hl,'C_MASS':session.cm}
        return records,features

In [ ]:
# Read L13, decide once, write only future layers within the same prefill

class RoutingSession:
    def __init__(self,engine,pkt,reader,policy,vectors,force_action,capture):
        self.e=engine;self.pkt=pkt;self.reader=reader;self.policy=policy;self.vectors=vectors
        self.capture=bool(capture);self.handles=[];self.phase='not_started'
        self.forward_calls=0;self.prefill_calls=0;self.route_calls=0;self.decode_writes=0
        self.actions=np.full(len(pkt['tokens']),int(force_action),int);self.probs=None
        if not np.isin(self.actions,[0,1,2]).all():raise ValueError('Action IDs must be 0/1/2.')
        self.hq=self.hl=self.cm=None;self.qraw=self.kraw=self.rope=None
        self.proposed_ratios=np.zeros(len(self.actions));self.actual_ratios=np.zeros(len(self.actions))
        self.proposed_max=0.;self.trace=[];self.ready=reader is None
    def enter_forward(self,mod,args,kwargs):
        self.forward_calls+=1
        self.phase='prefill' if self.forward_calls==1 else 'decode'
        ids=kwargs.get('input_ids')
        if self.phase=='prefill':
            if ids.shape!=self.pkt['ids'].shape:raise AssertionError('Unexpected initial input shape.')
            self.prefill_calls+=1;self.trace.append('prefill')
        elif ids.shape[1]!=1:raise AssertionError('Prompt replay detected during decoding.')
        elif kwargs.get('past_key_values') is None:raise AssertionError('Decoding must reuse the prefill KV cache.')
    def capture_rope(self,mod,args,kwargs):
        if self.phase=='prefill':self.rope=kwargs.get('position_embeddings')
    def capture_q(self,mod,args,out):
        if self.phase=='prefill':self.qraw=out.detach()
    def capture_k(self,mod,args,out):
        if self.phase=='prefill':self.kraw=out.detach()
    def read_head(self,mod,args):
        if self.phase!='prefill':return
        x=args[0];B,T,_=x.shape;h=x.reshape(B,T,self.e.nheads,self.e.hdim)
        self.hl=h[:,-1].detach().float().cpu().numpy()
        if self.capture:
            self.hq=h[torch.arange(B,device=x.device),self.pkt['qidx']].detach().float().cpu().numpy()
            self.cm=self.context_mass()
        self.trace.append(f'L{self.e.read_layer}_head_read')
    def context_mass(self):
        # Offline head discovery only; reuse actual Q/K projections, never recompute them.
        from transformers.models.llama.modeling_llama import apply_rotary_pos_emb,repeat_kv
        if self.rope is None or self.qraw is None or self.kraw is None:raise RuntimeError('Missing Llama Q/K/RoPE.')
        B,T,_=self.qraw.shape
        q=self.qraw.reshape(B,T,self.e.nheads,self.e.hdim).transpose(1,2)
        k=self.kraw.reshape(B,T,self.e.nkv,self.e.hdim).transpose(1,2)
        q,k=apply_rotary_pos_emb(q,k,*self.rope);k=repeat_kv(k,self.e.nheads//self.e.nkv)
        scores=torch.einsum('bhd,bhkd->bhk',q[:,:,-1].float(),k.float())*(self.e.hdim**-.5)
        prob=scores.masked_fill(self.pkt['mask'][:,None,:].eq(0),-1e9).softmax(-1)
        cm=torch.stack([prob[j,:,ix].sum(-1) for j,ix in enumerate(self.pkt['cidx'])])
        self.qraw=self.kraw=None
        return cm.cpu().numpy()
    def decide(self,mod,args,out):
        if self.phase!='prefill':return out
        if self.reader is not None:
            self.route_calls+=1;self.probs=read_probs(self.reader,self.hl)
            self.actions=choose_actions(self.policy['router'],self.probs,self.policy['guard'])
            self.trace.append(f'L{self.e.read_layer}_route')
        self.ready=True
        return out
    def write(self,ll):
        def hook(mod,args,out):
            if self.phase!='prefill':return out
            if not self.ready:raise AssertionError('Read-before-write violation.')
            self.trace.append(f'L{ll}_write_check')
            if not np.any(self.actions):return out
            if self.vectors is None or ll not in self.vectors:raise ValueError('Missing frozen action vector.')
            h=out[0] if isinstance(out,tuple) else out;x=h[:,-1].float()
            v=torch.as_tensor(self.vectors[ll],dtype=torch.float32,device=h.device)
            if v.shape!=(h.shape[-1],):raise ValueError('Action vector shape mismatch.')
            signs=torch.tensor(np.where(self.actions==2,1.,np.where(self.actions==1,-1.,0.)),dtype=torch.float32,device=h.device)
            proposed=signs[:,None]*self.e.alpha*v
            scale=(self.e.cap*x.norm(dim=-1,keepdim=True)/proposed.norm(dim=-1,keepdim=True).clamp_min(1e-12)).clamp(max=1.)
            delta=proposed*scale
            ratios=delta.norm(dim=-1)/x.norm(dim=-1).clamp_min(1e-12)
            self.proposed_ratios=np.maximum(self.proposed_ratios,ratios.cpu().numpy())
            self.proposed_max=float(self.proposed_ratios.max())
            new=h.clone();new[:,-1]=h[:,-1]+delta.to(h.dtype)
            actual=(new[:,-1].float()-x).norm(dim=-1)/x.norm(dim=-1).clamp_min(1e-12)
            self.actual_ratios=np.maximum(self.actual_ratios,actual.cpu().numpy())
            return (new,)+out[1:] if isinstance(out,tuple) else new
        return hook
    def __enter__(self):
        layers=self.e.model.model.layers
        self.handles.append(self.e.model.model.register_forward_pre_hook(self.enter_forward,with_kwargs=True))
        if self.reader is not None or self.capture:
            attn=layers[self.e.read_layer].self_attn
            self.handles.append(attn.o_proj.register_forward_pre_hook(self.read_head))
            if self.capture:
                self.handles.append(attn.register_forward_pre_hook(self.capture_rope,with_kwargs=True))
                self.handles.append(attn.q_proj.register_forward_hook(self.capture_q))
                self.handles.append(attn.k_proj.register_forward_hook(self.capture_k))
            self.handles.append(layers[self.e.read_layer].register_forward_hook(self.decide))
        for ll in self.e.write_layers:self.handles.append(layers[ll].register_forward_hook(self.write(ll)))
        return self
    def __exit__(self,*exc):
        for h in self.handles:h.remove()
        self.handles=[];self.qraw=self.kraw=self.rope=None

In [ ]:
# Offline instruction-vector fitting — inherited E14 recipe, no Packed

@torch.inference_mode()
def collect_instruction_residuals(engine,inputs,directive,batch_size=2):
    vectors={l:[] for l in engine.write_layers}
    for start in range(0,len(inputs),batch_size):
        pkt=engine.batch(inputs[start:start+batch_size],directive);captured={};handles=[]
        def make_hook(l):
            def hook(mod,args,out):
                h=out[0] if isinstance(out,tuple) else out
                captured[l]=h[:,-1].detach().float().cpu().numpy()
            return hook
        try:
            for l in engine.write_layers:handles.append(engine.model.model.layers[l].register_forward_hook(make_hook(l)))
            engine.model.model(input_ids=pkt['ids'],attention_mask=pkt['mask'],position_ids=pkt['pos'],use_cache=False,return_dict=True)
        finally:
            for h in handles:h.remove()
        for l in vectors:vectors[l].append(captured[l])
    return {l:np.concatenate(v) for l,v in vectors.items()}

def fit_instruction_vectors(engine,inputs,batch_size=2):
    ctx=collect_instruction_residuals(engine,inputs,'context',batch_size)
    mem=collect_instruction_residuals(engine,inputs,'memory',batch_size)
    out={}
    for l in engine.write_layers:
        diff=(ctx[l].mean(0)-mem[l].mean(0)).astype(np.float64);norm=float(np.linalg.norm(diff))
        if norm<=1e-12:raise ValueError('Degenerate instruction direction.')
        unit=(diff/norm).astype(np.float32)
        out[l]=unit*max(float((ctx[l]@unit).std()),1e-5)
    return out

## 10. 원본 자동 확보 및 분할 검증
원본 데이터 해시를 고정합니다. 문자열 `Nan`을 결측치로 제거하지 않습니다. 외부 데이터를 K+만 남기도록 필터링하지 않습니다. 원본 파일/과거 실험 출력은 덮어쓰지 않습니다.

In [ ]:
from google.colab import drive,auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth
from io import FileIO
from IPython.display import display
import importlib.metadata as package_metadata

drive.mount('/content/drive')
ROOT=Path(CFG['OUTPUT_PARENT'])/CFG['RUN_TAG'];ROOT.mkdir(parents=True,exist_ok=True)
DATA=ROOT/'sources';DATA.mkdir(exist_ok=True)
SOURCES={
 'p8s':{'id':'1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr','name':'p8s_behavior_v2.csv',
        'sha256':'fd37b0afd04972e3834ffa5f0af7f362affcfac8e20325117d37d8de15447c1c'},
 'Klabels':{'id':'1LMrQ3R4XWTcBbDn3kk2mR1iTITYw-KFF','name':'k_relabel_v2.parquet',
        'sha256':'0833c2e593d1c37e18e1600034f84d541a7b767de0fa4dec28d8f21e3f0008fb'},
 'NQ-Swap':{'id':'1w4fgNYMaXYQ5oofJlnQQsLq_cbOeM3Qx','name':'nqswap.parquet',
        'sha256':'3414224407cd3fa806893e828ebd7f56362b571e45aa25fd653f0e6383370ae1'},
 'ConFiQA-QA':{'id':'1HlTM0EdhHKhdo7wT1muWPZCZmDqp3Ch-','name':'ConFiQA-QA.json',
        'sha256':'9b201a7e2f2bbf8ad8ae3364eb87c194cf5d6007789c8b5302c091fa8fae74e4'},
}
SERVICE=None
def obtain_source(name):
    global SERVICE
    spec=SOURCES[name];path=DATA/spec['name']
    if not path.exists():
        if SERVICE is None:
            auth.authenticate_user()
            creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
            SERVICE=build('drive','v3',credentials=creds,cache_discovery=False)
        temp=path.with_name(path.name+'.downloading')
        with FileIO(str(temp),'wb') as f:
            d=MediaIoBaseDownload(f,SERVICE.files().get_media(fileId=spec['id']))
            done=False
            while not done:_,done=d.next_chunk()
        if file_sha(temp)!=spec['sha256']:raise ValueError('Downloaded source hash differs: '+name)
        temp.replace(path)
    if file_sha(path)!=spec['sha256']:raise ValueError('Source hash differs: '+name)
    return path

F=load_p8s(obtain_source('p8s'),obtain_source('Klabels'))
SP=e14_split(F,CFG['FOLD'],CFG['SEED'])
# Validate against existing E17 split when present; no existing model is reused.
old=Path(CFG['OUTPUT_PARENT'])/'E17_geometry_matched_heads_v1'/f"fold_{CFG['FOLD']}"/'READOUT_LOCK.json'
if old.exists():
    lock=json.loads(old.read_text())
    for field,role in [('fit_rows','fit'),('pc_rows','policy_cal'),('test_rows','test')]:
        assert np.array_equal(np.asarray(lock[field],int),np.flatnonzero(SP.role.eq(role))),field
F['qid']=F.question.astype(str);F['variant']=np.where(F.c.eq(1),'C+','C-');F['context']=F.ctx_text
F['role']=SP.role
TRAIN=F[F.role.isin(['fit','utility_fit'])].copy().reset_index(drop=True)
CAL=F[F.role.eq('policy_cal')].copy().reset_index(drop=True)
P8S_TEST=F[F.role.eq('test')].copy().reset_index(drop=True)
for a,b in itertools.combinations([TRAIN,CAL,P8S_TEST],2):
    assert not set(a.question.map(question_key)) & set(b.question.map(question_key))
assert (TRAIN.role.eq('fit').sum(),TRAIN.role.eq('utility_fit').sum(),len(CAL),len(P8S_TEST))==(732,122,124,246)
SP.to_csv(ROOT/'SPLIT_MANIFEST.csv',index=False)
display(SP.groupby('role').agg(questions=('question','nunique'),inputs=('row','size')))
print('동일 분할 검증 완료. 모든 기존 결과는 탐색 이력이 있는 자료입니다.')

In [ ]:
# 11. 모델 로드 — 재실행 시 중복 메모리 할당 금지
from transformers import AutoTokenizer,AutoModelForCausalLM
from huggingface_hub import get_token,login
from getpass import getpass
import transformers
assert transformers.__version__=='4.57.1'
signature={'model':CFG['MODEL'],'revision':CFG['REVISION'],'dtype':'bfloat16','backend':'sdpa'}
if globals().get('MODEL') is not None:
    if globals().get('MODEL_SIGNATURE')!=signature:raise RuntimeError('다른 모델이 이미 있습니다. 새 런타임이 필요합니다.')
else:
    if torch.cuda.mem_get_info()[0]/2**30 < 25:raise RuntimeError('기존 모델 메모리가 남았는지 확인하세요. 새 런타임이 필요합니다.')
    hf=get_token() or os.environ.get('HF_TOKEN')
    if not hf:
        try:
            from google.colab import userdata
            hf=userdata.get('HF_TOKEN')
        except Exception:pass
    if not hf:hf=getpass('Llama 접근 가능한 Hugging Face Read token: ').strip()
    if not hf:raise RuntimeError('HF_TOKEN이 필요합니다.')
    login(token=hf,add_to_git_credential=False)
    TOK=AutoTokenizer.from_pretrained(CFG['MODEL'],revision=CFG['REVISION'],token=hf,use_fast=True)
    TOK.padding_side='left'
    if TOK.pad_token_id is None:TOK.pad_token=TOK.eos_token
    MODEL=AutoModelForCausalLM.from_pretrained(CFG['MODEL'],revision=CFG['REVISION'],token=hf,
          dtype=torch.bfloat16,device_map={'':'cuda:0'},attn_implementation='sdpa').eval()
    for p in MODEL.parameters():p.requires_grad_(False)
    MODEL_SIGNATURE=signature;del hf
assert MODEL.config.num_attention_heads==32 and MODEL.config.hidden_size==4096
assert next(MODEL.parameters()).dtype==torch.bfloat16
ENGINE=LLMEngine(MODEL,TOK,CFG['READ_LAYER'],CFG['WRITE_LAYERS'],CFG['MAX_PROMPT_TOKENS'],
                 CFG['MAX_NEW_TOKENS'],CFG['ALPHA'],CFG['DELTA_CAP'])
torch.manual_seed(CFG['SEED']);np.random.seed(CFG['SEED']);torch.backends.cuda.matmul.allow_tf32=False
ENV={k:package_metadata.version(k) for k in ['torch','transformers','numpy','pandas','scikit-learn','pyarrow','accelerate']}
ENV.update({'gpu':torch.cuda.get_device_name(0),'cuda':torch.version.cuda})
print('Loaded:',CFG['REVISION'],'|',ENV)

## 12. 캐시 및 실행 무결성
캐시는 모델·프롬프트·원본·설정·정책·코드에 묶습니다. 한 질문을 실제 생성하는 함수는 `PromptInput(question, context)`만 받으며, gold·variant·K는 받지 않습니다. 재실행 때는 같은 RUN_TAG로 재개하고, 코드를 바꾸면 새 RUN_TAG를 사용합니다.

In [ ]:
def stable_code(fn):
    fn=inspect.unwrap(fn)
    def walk(x):
        if isinstance(x,types.CodeType):
            return {'bytecode':x.co_code.hex(),'consts':[walk(v) for v in x.co_consts],
                    'names':x.co_names,'vars':x.co_varnames,'args':x.co_argcount,'kwonly':x.co_kwonlyargcount}
        if isinstance(x,bytes):return {'bytes':x.hex()}
        if isinstance(x,(tuple,list)):return [walk(v) for v in x]
        if x is None or isinstance(x,(str,int,float,bool)):return x
        return str(type(x))
    return walk(fn.__code__)

def public_inputs(meta):
    return [PromptInput(str(r.question),str(r.context)) for r in meta.itertuples()]

def cached_generate(meta,tag,reader=None,policy=None,force_action=0,capture=False,closedbook=False):
    # Join labels later. Bookkeeping fields are never passed into ENGINE.generate.
    folder=ROOT/'generation'/tag;folder.mkdir(parents=True,exist_ok=True)
    inputs=[PromptInput(str(r.question),None if closedbook else str(r.context)) for r in meta.itertuples()]
    public=[{'question':p.question,'context':p.context} for p in inputs]
    fp=digest({'run':RUN_HASH,'inputs':public,'reader':reader,'policy':policy,'force':force_action,
               'vectors':VECTOR_HASH,'capture':capture,'offline_closedbook':closedbook})
    lock_json(folder/'MANIFEST.json',{'fingerprint':fp,'rows':len(inputs)})
    records=[];banks={k:[] for k in ('HEAD_Q','HEAD_LAST','C_MASS')}
    for start in range(0,len(inputs),CFG['BATCH_SIZE']):
        batch=inputs[start:start+CFG['BATCH_SIZE']]
        jp=folder/f'batch_{start:06d}.json';ap=folder/f'batch_{start:06d}.npz'
        if jp.exists():
            obj=json.loads(jp.read_text())
            if obj['fingerprint']!=fp or obj['start']!=start or len(obj['records'])!=len(batch):raise ValueError('Generation cache mismatch.')
            if digest(obj['records'])!=obj['records_sha']:raise ValueError('Cached records were modified.')
            rec=obj['records']
            if capture:
                if file_sha(ap)!=obj['activation_sha']:raise ValueError('Activation cache modified.')
                with np.load(ap,allow_pickle=False) as z:bank={k:z[k].copy() for k in banks}
        else:
            rec,bank=ENGINE.generate(batch,reader=reader,policy=policy,vectors=VECTORS,
                                     force_action=force_action,capture=capture)
            obj={'fingerprint':fp,'start':start,'records':rec,'records_sha':digest(rec),
                 'purpose':'offline_K_label_only' if closedbook else 'context_generation'}
            if capture:
                tmp=ap.with_name(ap.name+'.tmp')
                with tmp.open('wb') as f:np.savez_compressed(f,**bank)
                tmp.replace(ap);obj['activation_sha']=file_sha(ap)
            atomic_json(jp,obj)
        records.extend(rec)
        if capture:
            for k in banks:banks[k].append(bank[k])
        if start==0 or (start//CFG['BATCH_SIZE']+1)%50==0:
            print(tag,min(start+len(batch),len(inputs)),'/',len(inputs),flush=True)
    if len(records)!=len(meta):raise AssertionError('Incomplete generation.')
    return records,({k:np.concatenate(v) for k,v in banks.items()} if capture else None)

names=['role_heads','fit_linear','predict_linear','fit_readout','read_probs','oof_readout',
       'routing_features','fit_outcome_router','outcome_scores','choose_actions','calibrate_policy',
       'e14_split','norm_answer','answer_em','raw_em','answer_extract','alias_list','parse_gold',
       'load_p8s','state_ids','check_probs','preservation','cluster_delta_ci','evaluate_outputs',
       'cached_generate','public_inputs','fit_instruction_vectors','collect_instruction_residuals']
implement={n:stable_code(globals()[n]) for n in names}
for cls in (LLMEngine,RoutingSession):
    implement[cls.__name__]={n:stable_code(v) for n,v in cls.__dict__.items() if inspect.isfunction(v)}
RUN_MANIFEST={'cfg':CFG,'runtime':ENV,'sources':SOURCES,'split':SP.to_dict('records'),
              'system_prompt':SYSTEM_PROMPT,'cb_system_prompt':CB_SYSTEM_PROMPT,
              'chat_template_hash':digest(TOK.chat_template),'code_hash':digest(implement),
              'model_config_hash':digest(MODEL.config.to_dict()),
              'protocol':'exploratory; no Packed; original HEAD_LAST readout; OOF gain/harm router'}
lock_json(ROOT/'RUN_LOCK.json',RUN_MANIFEST);RUN_HASH=digest(RUN_MANIFEST)

In [ ]:
# 13. 고정 개입 수단 — E14의 instruction contrast를 fit 질문에서만 새로 추출
fit_questions=np.array(sorted(TRAIN.loc[TRAIN.role.eq('fit'),'question'].unique()))
rng=np.random.default_rng(CFG['SEED']+7000+CFG['FOLD']);rng.shuffle(fit_questions)
chosen=set(fit_questions[:CFG['DIRECTION_QUESTIONS']])
direction_meta=TRAIN[TRAIN.role.eq('fit') & TRAIN.question.isin(chosen)].copy()
vp=ROOT/'INSTRUCTION_VECTORS.npz';vm=ROOT/'INSTRUCTION_VECTORS_MANIFEST.json'
vmeta={'run':RUN_HASH,'source_questions':sorted(chosen),'rows':direction_meta.row.astype(int).tolist(),
       'fit_only':True,'recipe':'E14: unit(mean(context)-mean(memory))*std(context projection)',
       'phase':'prefill-last-token only','alpha':CFG['ALPHA'],'cap':CFG['DELTA_CAP']}
if vp.exists() and vm.exists():
    old=json.loads(vm.read_text());assert old['spec']==vmeta and old['sha256']==file_sha(vp)
    with np.load(vp,allow_pickle=False) as z:VECTORS={l:z[f'L{l}'].copy() for l in CFG['WRITE_LAYERS']}
else:
    VECTORS=fit_instruction_vectors(ENGINE,public_inputs(direction_meta),CFG['BATCH_SIZE'])
    tmp=vp.with_name(vp.name+'.tmp')
    with tmp.open('wb') as f:np.savez_compressed(f,**{f'L{l}':v for l,v in VECTORS.items()})
    tmp.replace(vp);atomic_json(vm,{'spec':vmeta,'sha256':file_sha(vp)})
VECTOR_HASH=file_sha(vp)
print('고정 벡터: train',len(chosen),'질문. 온라인 closed-book / Packed 없음.')

In [ ]:
# 14. train/cal 무개입 실행: head bank와 실제 Base 답을 함께 수집
TRAIN_BASE,TBANK=cached_generate(TRAIN,'train/Base',capture=True)
CAL_BASE,CBANK=cached_generate(CAL,'policy_cal/Base',capture=True)
for bank,meta in [(TBANK,TRAIN),(CBANK,CAL)]:
    assert bank['HEAD_LAST'].shape==(len(meta),32,128)
    assert bank['HEAD_Q'].shape==(len(meta),32,128) and bank['C_MASS'].shape==(len(meta),32)
    assert all(np.isfinite(v).all() for v in bank.values())
fi=np.flatnonzero(TRAIN.role.eq('fit'));uf=np.flatnonzero(TRAIN.role.eq('utility_fit'))
READER=fit_readout(TBANK['HEAD_Q'][fi],TBANK['HEAD_LAST'][fi],TBANK['C_MASS'][fi],
                  TRAIN.state.to_numpy()[fi],CFG['ROLE_TOPK'],CFG['READOUT_C'],CFG['SEED'])
OOF,OOF_AUDIT=oof_readout(TBANK['HEAD_Q'][fi],TBANK['HEAD_LAST'][fi],TBANK['C_MASS'][fi],
                TRAIN.state.to_numpy()[fi],TRAIN.question.to_numpy()[fi],CFG['OOF_FOLDS'],
                CFG['SEED'],CFG['ROLE_TOPK'],CFG['READOUT_C'])
TRAIN_P=np.full((len(TRAIN),4),np.nan);TRAIN_P[fi]=OOF
TRAIN_P[uf]=read_probs(READER,TBANK['HEAD_LAST'][uf])
CAL_P=read_probs(READER,CBANK['HEAD_LAST'])
assert np.isfinite(TRAIN_P).all()
lock_json(ROOT/'READOUT_LOCK.json',READER);atomic_json(ROOT/'OOF_READOUT_AUDIT.json',OOF_AUDIT)
np.savez_compressed(ROOT/'READOUT_TRAIN_CAL_PROBABILITIES.npz',train_prob=TRAIN_P,
                    train_source_rows=TRAIN.row.to_numpy(),cal_prob=CAL_P,cal_source_rows=CAL.row.to_numpy())
print('선택 헤드:',READER['heads'],'| 역할별:',READER['roles'])
print('fit 질문 OOF Macro-F1:',f1_score(TRAIN.state.to_numpy()[fi],OOF.argmax(1),average='macro',labels=range(4)))
print('위 값은 상태판독 OOF이며, 최종 QA 성능이 아닙니다.')

## 15. 행동 라우터 학습: 후보는 학습용 정답표에만 사용
훈련 단계에서는 동일 입력에 no-op / retain / accept를 각각 적용해 실제 결과를 측정합니다. 이 **오프라인 행동 결과표**로 라우터를 학습하지만, 온라인 함수는 이 표를 받지 않습니다.
각 입력의 `G=Base 오답→개입 정답`, `H=Base 정답→개입 오답`을 구분합니다. 미관측/희소 사건은 smoothing과 최소 회복 수 기준으로 처리합니다.
같은 판독확률·개입 벡터 아래 P4, K-only, C-only, conflict-only를 비교합니다.

In [ ]:
TRAIN_R,_=cached_generate(TRAIN,'train/retain',force_action=1)
TRAIN_A,_=cached_generate(TRAIN,'train/accept',force_action=2)
CAL_R,_=cached_generate(CAL,'policy_cal/retain',force_action=1)
CAL_A,_=cached_generate(CAL,'policy_cal/accept',force_action=2)
def em_matrix(meta,arms):
    return np.column_stack([[answer_em(r['raw'],g) for r,g in zip(arm,meta.aliases)] for arm in arms])
TRAIN_EM=em_matrix(TRAIN,[TRAIN_BASE,TRAIN_R,TRAIN_A])
CAL_EM=em_matrix(CAL,[CAL_BASE,CAL_R,CAL_A])
CAL_ACTION_DIAGNOSTICS=[]
for label,e in [('Base',CAL_EM[:,0]),('Always_retain',CAL_EM[:,1]),
                ('Always_accept',CAL_EM[:,2]),('ActionOracle_DIAGNOSTIC_ONLY',CAL_EM.max(1))]:
    for subset,mask in {'all':np.ones(len(CAL),bool),'C+':CAL.c.eq(1).to_numpy(),
                        'C-':CAL.c.eq(0).to_numpy()}.items():
        CAL_ACTION_DIAGNOSTICS.append({'method':label,'subset':subset,
            'diagnostic_only':label.startswith('ActionOracle'),**preservation(CAL_EM[:,0],e,mask)})
pd.DataFrame(CAL_ACTION_DIAGNOSTICS).to_csv(ROOT/'CAL_ACTION_DIAGNOSTICS.csv',index=False)
# Closed-book only supplies calibration subset labels; it never supplies a route feature.
CAL_UNIQUE=CAL.drop_duplicates('question').copy()
CAL_CB,_=cached_generate(CAL_UNIQUE,'policy_cal/offline_closedbook',closedbook=True)
cb_raw=dict(zip(CAL_UNIQUE.question,[r['raw'] for r in CAL_CB]))
CAL_K=np.array([answer_em(cb_raw[q],g) for q,g in zip(CAL.question,CAL.aliases)],bool)
ROUTERS={mode:fit_outcome_router(TRAIN_P,TRAIN_EM,mode,CFG['ROUTER_C'],CFG['HARM_WEIGHT'],
                               CFG['MIN_TRAIN_GAINS'],CFG['SEED'])
         for mode in ['P4','K_only','C_only','Conflict_only']}
POLICIES={'4state_rule':{'router':{'kind':'rule'},'guard':{'force_noop':False,'confidence':0.}}}
GRIDS=[]
for mode,router in [('4state_confidence',{'kind':'rule'}),*ROUTERS.items()]:
    guard,grid=calibrate_policy(router,CAL_P,CAL_EM,CAL.c.to_numpy()==1,CAL_K,
          CFG['MARGIN_GRID'],CFG['HARM_GRID'],questions=CAL.question.tolist())
    name='S4R_gain_harm' if mode=='P4' else mode
    POLICIES[name]={'router':router,'guard':guard}
    grid.insert(0,'method',name);GRIDS.append(grid)
GRID=pd.concat(GRIDS,ignore_index=True);GRID.to_csv(ROOT/'FULL_POLICY_CAL_GRID.csv',index=False)
TARGETS=TRAIN[['row','qid','question','variant','role']].copy()
TARGETS['pstate']=TRAIN_P.tolist()
for i,a in enumerate(ACTIONS):TARGETS[a+'_em']=TRAIN_EM[:,i]
TARGETS.to_json(ROOT/'OFFLINE_ROUTER_TRAIN_TARGETS.jsonl',orient='records',lines=True,force_ascii=False)
BUNDLE={'reader':READER,'policies':POLICIES,'run_hash':RUN_HASH,'vector_sha':VECTOR_HASH,
        'class_order':STATES,'action_order':ACTIONS,'primary_method':CFG['PRIMARY_METHOD'],
        'claim':'single prompt prefill and single decoding trajectory; additional offline training/evaluation excluded',
        'readout_fit_questions':sorted(TRAIN.loc[TRAIN.role.eq('fit'),'question'].unique()),
        'router_fit_questions':sorted(TRAIN.question.unique()),'calibration_questions':sorted(CAL.question.unique())}
lock_json(ROOT/'ROUTER_LOCK.json',BUNDLE);BUNDLE_HASH=digest(BUNDLE)
display(GRID[GRID.selected][['method','reason' if 'reason' in GRID else 'eligible','correct','lost',
                  'gained','Cplus_lost','KminusCplus_lost','switches']])
print('PRIMARY:',POLICIES[CFG['PRIMARY_METHOD']]['guard'])
print('정책 잠금 완료. 뒤의 test에서 강도/임계값/방법을 다시 고르지 않습니다.')

In [ ]:
# 16. GPU wiring smoke: 동일 프롬프트에서 no-op 라우터가 Base를 정확히 유지해야 함
smoke=CAL.iloc[:CFG['BATCH_SIZE']].copy()
noop_policy={'router':{'kind':'rule'},'guard':{'force_noop':True}}
actual,_=ENGINE.generate(public_inputs(smoke),reader=READER,policy=noop_policy,vectors=VECTORS)
for i,r in enumerate(actual):
    assert r['token_ids']==CAL_BASE[i]['token_ids'],'No-op identity failure: do not trust the experiment.'
    assert r['prefill_calls']==1 and r['route_calls']==1 and r['decode_writes']==0
    assert np.allclose(r['pstate'],CAL_P[i],atol=1e-6,rtol=1e-5),'Offline/live readout mismatch.'
    assert r['trace'].index('L13_route')<r['trace'].index('L14_write_check')
atomic_json(ROOT/'LIVE_SMOKE.json',actual)
print('실제 GPU smoke 통과: Base 동일, 1 prefill, 1 route, read-before-write, decode 개입 0.')

## 17. 외부 입력 구성 — 같은 원본 ID, 질문 반복은 별도 기록
NQ는 기존 E20처럼 corpus subset 앞 1,000 원본 ID, ConFiQA는 질문 중복 제거 후 앞 1,000개를 사용합니다. E19의 SHA 표본 500개와는 같은 평가셋이 아닙니다. 잘못된/모호한 별칭은 기록하거나 중단하며, 모델 성능을 보고 표본을 제외하지 않습니다.
NQ의 같은 질문·다른 ID는 원본 가중 평균에 유지하고, 신뢰구간은 **정규화 질문 문자열**로 묶습니다. train/cal 중복 질문을 제외한 공통 보조 표도 저장합니다.

In [ ]:
def external_row(ds,qid,q,cp,cn,gold,decoy):
    gold=[v for v in alias_list(gold) if norm_answer(v)]
    decoy=[v for v in alias_list(decoy) if norm_answer(v)]
    if not all(isinstance(v,str) and v.strip() for v in [q,cp,cn]) or not gold or not decoy:
        raise ValueError('Invalid selected external input: '+str(qid))
    if set(map(norm_answer,gold)) & set(map(norm_answer,decoy)):
        raise ValueError('Gold/decoy aliases overlap: '+str(qid))
    return [{'dataset':ds,'qid':str(qid),'question':q,'context':cx,'variant':v,
             'c':c,'aliases':gold,'decoy':decoy} for v,c,cx in [('C+',1,cp),('C-',0,cn)]]

nq=pd.read_parquet(obtain_source('NQ-Swap'))
nq=nq[nq.substitution_type.eq('corpus')].reset_index(drop=True).head(CFG['NQ_QUESTIONS'])
if len(nq)!=CFG['NQ_QUESTIONS']:raise ValueError('Insufficient NQ original IDs.')
nrows=[]
for r in nq.itertuples():
    nrows+=external_row('NQ-Swap',r.id,r.question,r.original_context,r.substituted_context,
                        r.original_answers,r.substituted_answers)
cq=pd.read_json(obtain_source('ConFiQA-QA'))
cq=cq.drop_duplicates('question',keep='first').dropna(subset=['question','orig_context','cf_context','orig_answer']).head(CFG['CONFIQA_QUESTIONS'])
if len(cq)!=CFG['CONFIQA_QUESTIONS']:raise ValueError('Insufficient ConFiQA questions.')
crows=[]
for r in cq.itertuples():
    q=str(r.question).strip();qid=hashlib.md5(q.encode()).hexdigest()[:12]
    crows+=external_row('ConFiQA-QA',qid,q,r.orig_context,r.cf_context,
                        alias_list(r.orig_answer)+alias_list(r.orig_alias),r.cf_answer)
TESTSETS={'p8s':P8S_TEST.copy(),'NQ-Swap':pd.DataFrame(nrows),'ConFiQA-QA':pd.DataFrame(crows)}
used={question_key(q) for q in list(TRAIN.question)+list(CAL.question)}
used_loose={norm_answer(q) for q in list(TRAIN.question)+list(CAL.question)}
INPUT_AUDIT=[]
for ds,meta in TESTSETS.items():
    assert not meta.duplicated(['qid','variant']).any()
    assert meta.groupby('qid').variant.nunique().eq(2).all()
    meta['seen_train_cal']=meta.question.map(question_key).isin(used)|meta.question.map(norm_answer).isin(used_loose)
    # Validate every prompt before generating any test output; never silently truncate/drop it.
    for item in public_inputs(meta):ENGINE.render(item)
    INPUT_AUDIT.append({'dataset':ds,'original_ids':meta.qid.nunique(),
                        'unique_question_clusters':meta.question.map(question_key).nunique(),
                        'rows':len(meta),'overlap_rows':int(meta.seen_train_cal.sum())})
    if ds=='p8s':assert not meta.seen_train_cal.any()
lock_json(ROOT/'TEST_INPUT_LOCK.json',{'datasets':{d:m[['qid','variant','question','context','aliases']].to_dict('records')
          for d,m in TESTSETS.items()},'policy_lock_hash':BUNDLE_HASH})
atomic_json(ROOT/'EVALUATION_INPUT_AUDIT.json',INPUT_AUDIT);display(pd.DataFrame(INPUT_AUDIT))

## 18. 테스트는 실제 온라인 라우터를 실행
Base/고정 개입/규칙/학습 라우터 모두 동일한 프롬프트·모델·개입 벡터·길이·스케줄입니다.
`S4R_gain_harm` 결과를 후보 표에서 골라서 만들지 않습니다. 각 입력의 L13 hook이 그 실행 안에서 직접 행동을 정합니다.
전체 벤치마크는 비교 방법만큼 여러 번 실행되지만, **각 방법의 개별 추론**은 한 prefill/한 생성 경로입니다.

In [ ]:
assert digest(json.loads((ROOT/'ROUTER_LOCK.json').read_text()))==BUNDLE_HASH
ALL_OUTPUTS={};SUMMARY=[];CHANGES=[];ORACLES=[];READOUT_METRICS=[];UNSEEN=[]
for ds,meta in TESTSETS.items():
    out={}
    out['Base'],_=cached_generate(meta,ds+'/Base')
    # Actual routed generations occur independently; no access to a test candidate bank.
    for method,policy in POLICIES.items():
        out[method],_=cached_generate(meta,ds+'/'+method,reader=READER,policy=policy)
    out['Always_retain'],_=cached_generate(meta,ds+'/Always_retain',force_action=1)
    out['Always_accept'],_=cached_generate(meta,ds+'/Always_accept',force_action=2)
    # After all contextual methods, obtain current K labels in an isolated evaluator.
    unique=meta.drop_duplicates('question').copy()
    cbrecs,_=cached_generate(unique,ds+'/offline_closedbook_labels',closedbook=True)
    cb=dict(zip(unique.question,[r['raw'] for r in cbrecs]))
    known=np.array([answer_em(cb[q],g) for q,g in zip(meta.question,meta.aliases)],bool)
    table,changes,base,y=evaluate_outputs(meta,out,known,CFG['BOOTSTRAP'],CFG['SEED'])
    table.insert(0,'dataset',ds);SUMMARY.append(table)
    changes.insert(0,'dataset',ds);CHANGES.append(changes)
    ALL_OUTPUTS[ds]=out
    ep=ROOT/'evaluation'/ds;ep.mkdir(parents=True,exist_ok=True)
    table.to_csv(ep/'SUMMARY.csv',index=False);changes.to_csv(ep/'CHANGED_CASES.csv',index=False)
    contrasts=[]
    primary_em=np.array([answer_em(r['raw'],g) for r,g in zip(out[CFG['PRIMARY_METHOD']],meta.aliases)])
    for reference in ['Base','Always_retain','Always_accept','4state_rule','4state_confidence','K_only','C_only','Conflict_only']:
        refem=np.array([answer_em(r['raw'],g) for r,g in zip(out[reference],meta.aliases)])
        for subset,mask in {'all':np.ones(len(meta),bool),'C+':meta.c.eq(1).to_numpy(),
                            'K-C+':y==3,'K+C-':y==1}.items():
            d,lo,hi=cluster_delta_ci(refem,primary_em,meta.question,mask,CFG['BOOTSTRAP'],CFG['SEED'])
            contrasts.append({'primary':CFG['PRIMARY_METHOD'],'reference':reference,'subset':subset,
                'delta_EM':d,'CI_low':lo,'CI_high':hi,
                'interval_scope':'paired normalized-question clusters; conditional on frozen policy; not multiplicity-adjusted',
                **preservation(refem,primary_em,mask)})
    pd.DataFrame(contrasts).to_csv(ep/'ROUTER_CONTRASTS.csv',index=False)
    p=np.array([r['pstate'] for r in out[CFG['PRIMARY_METHOD']]])
    targets={'current_standalone_CB_K':y}
    if ds=='p8s':targets['historical_corrected_K']=meta.state.to_numpy()
    for label_source,target in targets.items():
        READOUT_METRICS.append({'dataset':ds,'K_definition':label_source,
            'Macro_F1':float(f1_score(target,p.argmax(1),labels=range(4),average='macro',zero_division=0)),
            'K_AUROC':float(roc_auc_score(np.isin(target,[0,1]),p[:,0]+p[:,1])) if np.unique(np.isin(target,[0,1])).size==2 else None,
            'C_AUROC':float(roc_auc_score(np.isin(target,[0,3]),p[:,0]+p[:,3])),
            'confusion':confusion_matrix(target,p.argmax(1),labels=range(4)).tolist()})
    action_em=em_matrix(meta,[out['Base'],out['Always_retain'],out['Always_accept']])
    oracle_action=action_em.max(1)
    true_state_actions=np.where(y==1,1,np.where(y==3,2,0))
    oracle_state=action_em[np.arange(len(meta)),true_state_actions]
    actual=np.array([answer_em(r['raw'],g) for r,g in zip(out[CFG['PRIMARY_METHOD']],meta.aliases)])
    for label,e in [('ActionOracle_fixed_3actions',oracle_action),('StateOracle_rule_NOT_upper_bound',oracle_state)]:
        for subset,mask in {'all':np.ones(len(meta),bool),'C+':meta.c.eq(1).to_numpy(),
                            'K-C+':y==3,'K+C-':y==1}.items():
            ORACLES.append({'dataset':ds,'diagnostic':label,'subset':subset,'diagnostic_only':True,
                            **preservation(base,e,mask)})
    pd.DataFrame({'qid':meta.qid,'variant':meta.variant,'current_K':known.astype(int),
                  'source_K':meta.k if ds=='p8s' else None,'Base_em':base,'S4R_em':actual,
                  'action_oracle_em':oracle_action,'missed_available_correct':(oracle_action>actual),
                  'selected_action':[r['action'] for r in out[CFG['PRIMARY_METHOD']]]}).to_csv(ep/'ROUTER_REGRET_AND_K_AUDIT.csv',index=False)
    keep=~meta.seen_train_cal.to_numpy()
    submeta=meta.loc[keep].reset_index(drop=True)
    if len(submeta):
        subout={n:[r for r,k in zip(recs,keep) if k] for n,recs in out.items()}
        sub,_,_,_=evaluate_outputs(submeta,subout,known[keep],CFG['BOOTSTRAP'],CFG['SEED'])
        sub.insert(0,'dataset',ds);UNSEEN.append(sub)
    # Export every online trace so prefill / route / write claims can be audited.
    traces=[]
    for method,recs in out.items():
        for m,r in zip(meta.itertuples(),recs):
            assert r['prefill_calls']==1 and r['input_streams']==1 and r['generation_trajectories']==1
            assert r['online_closedbook_calls']==0 and r['decode_writes']==0
            if method in POLICIES:assert r['route_calls']==1
            traces.append({'qid':str(m.qid),'variant':m.variant,'method':method,**r})
    pd.DataFrame(traces).to_json(ep/'ONLINE_OUTPUTS_AND_AUDIT.jsonl',orient='records',lines=True,force_ascii=False)
    # Compare actual routed answers against forced arms only AFTER evaluation, as a wiring audit.
    forced=[out['Base'],out['Always_retain'],out['Always_accept']]
    parity=[]
    for name in POLICIES:
        for i,r in enumerate(out[name]):
            a=ACTIONS.index(r['action']);target=forced[a][i]
            if r['token_ids']!=target['token_ids']:
                parity.append({'method':name,'qid':str(meta.iloc[i].qid),'variant':meta.iloc[i].variant,
                               'action':r['action'],'routed_raw':r['raw'],'forced_raw':target['raw']})
    atomic_json(ep/'ROUTED_FORCED_PARITY_DIFFERENCES.json',parity)
    if parity:raise RuntimeError('Actual route differs from same-action execution. Stop and inspect; no oracle-gap claim is valid yet.')

pd.concat(SUMMARY,ignore_index=True).to_csv(ROOT/'SUMMARY.csv',index=False)
pd.concat(CHANGES,ignore_index=True).to_csv(ROOT/'CHANGED_CASES.csv',index=False)
pd.DataFrame(ORACLES).to_csv(ROOT/'ORACLE_DIAGNOSTICS_ONLY.csv',index=False)
atomic_json(ROOT/'STATE_READOUT_METRICS.json',READOUT_METRICS)
if UNSEEN:pd.concat(UNSEEN,ignore_index=True).to_csv(ROOT/'COMMON_UNSEEN_RESULTS.csv',index=False)
print('실제 온라인 생성 및 사후 진단 완료.')

## 19. 비용 측정 — 실제 batch=1, warm-up 후 반복
학습용 행동 생성·closed-book 평가·Oracle 계산은 배포 비용에 포함하지 않되, 존재 자체를 숨기지 않습니다.
아래는 실제 Base와 S4R의 개별 질의 측정입니다. 첫 토큰 준비 시간과 전체 생성 시간을 분리하며, 답변 길이가 다르면 전체 시간도 달라질 수 있습니다. 수치가 나오기 전 속도 향상을 주장하지 않습니다.

In [ ]:
bench_inputs=public_inputs(CAL.drop_duplicates('question').head(CFG['BENCHMARK_PROMPTS']))
for name in ['Base',CFG['PRIMARY_METHOD']]:
    kwargs={} if name=='Base' else {'reader':READER,'policy':POLICIES[name],'vectors':VECTORS}
    ENGINE.generate([bench_inputs[0]],**kwargs)
measurements=[]
for rep in range(CFG['BENCHMARK_REPEATS']):
    for i,item in enumerate(bench_inputs):
        order=['Base',CFG['PRIMARY_METHOD']]
        if (i+rep)%2:order.reverse()
        for name in order:
            kwargs={} if name=='Base' else {'reader':READER,'policy':POLICIES[name],'vectors':VECTORS}
            rec,_=ENGINE.generate([item],**kwargs)
            measurements.append({'method':name,'repeat':rep,'cal_prompt_index':i,**rec[0]})
BENCH=pd.DataFrame(measurements);BENCH.to_json(ROOT/'BATCH1_RUNTIME_BENCHMARK.jsonl',orient='records',lines=True,force_ascii=False)
BENCH_SUM=BENCH.groupby('method').agg(n=('batch_seconds','size'),
    median_seconds=('batch_seconds','median'),p95_seconds=('batch_seconds',lambda x:x.quantile(.95)),
    median_first_token_ready=('batch_ttft_seconds','median'),
    median_generated_tokens=('generated_tokens','median'),peak_allocated_bytes=('peak_allocated_bytes','max'),
    intervention_rate=('action',lambda x:(x!='noop').mean()))
BENCH_SUM.to_csv(ROOT/'RUNTIME_SUMMARY.csv');display(BENCH_SUM)

In [ ]:
# 20. 실제 결과 요약 / ZIP — 성공 여부를 과장하지 않음
SUM=pd.concat(SUMMARY,ignore_index=True)
display(SUM[SUM.subset.eq('all')].pivot(index='method',columns='dataset',values='EM'))
PRIMARY=SUM[SUM.method.eq(CFG['PRIMARY_METHOD'])]
display(PRIMARY[['dataset','subset','n','EM','lost','gained','net','preservation','intervention_rate','CI_low','CI_high']])
status={'primary_policy_reason':POLICIES[CFG['PRIMARY_METHOD']]['guard']['reason'],
        'all_noop_on_cal':bool(POLICIES[CFG['PRIMARY_METHOD']]['guard'].get('force_noop',False)),
        'performance_claim':'Read SUMMARY/PRESERVATION identity counts; no automatic success declaration.',
        'one_pass_definition':'one contextual prompt prefill + one autoregressive trajectory per deployed method',
        'limitations':['already-used benchmarks, exploratory follow-up','small 62-question calibration set',
                       'no test safety guarantee','OOF-to-final readout distribution shift remains possible',
                       'K is measured answer correctness, not absence/presence of all knowledge',
                       'fixed three-action Oracle is diagnostic only; cannot be reported as deployed accuracy'],
        'new_label_budget_note':'not a 20%-label replication; reader fit 366Q, router outcome training 427Q, calibration 62Q',
        'routing_features':'P4 only; K/C/conflict marginal ablations use reduced versions of same predicted state signal'}
atomic_json(ROOT/'INTERPRETATION.json',status)
if status['all_noop_on_cal']:
    print('주의: 주 정책이 no-op으로 잠겼습니다. 정답 보존은 개입 선별 성공이나 성능 개선 증거가 아닙니다.')
report=f"""# E21 S4R actual run

Primary: {CFG['PRIMARY_METHOD']}

{canonical(status)}

## Overall answer-EM
{SUM[SUM.subset.eq('all')].to_markdown(index=False)}

## Primary preservation
{PRIMARY.to_markdown(index=False)}

## Runtime
{BENCH_SUM.to_markdown()}
"""
(ROOT/'RUN_REPORT.md').write_text(report,encoding='utf-8')
zip_path=ROOT/'E21_S4R_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if not p.is_file() or p==zip_path or p.suffix in {'.tmp','.downloading'}:continue
        rel=p.relative_to(ROOT)
        if rel.parts[0]=='sources':continue # do not duplicate raw datasets in sharing ZIP
        z.write(p,str(rel))
print('결과 경로:',ROOT)
print('ZIP:',zip_path)

## 논문에서 검증할 주장
**기존 성과:** 역할별 선택 헤드의 단일 prefill 표현으로 K/C 4-state를 판독한다.

**이번 확장:** 동일 상태 신호를 이용해 행동의 회복·손상 가능성을 구분하고, 고정 개입 수단에서 retain / accept / no-op을 선택한다. 단일 입력·단일 prefill·단일 생성 경로에서 실행한다.

**필수 비교:** Base, 고정 retain/accept, 단순 4-state 규칙, 신뢰도 게이트, S4R 이득·손실 라우터, K-only/C-only/conflict-only.

**성공 조건:** 실제 온라인 QA 개선과 C+/K−C+ 기존 정답 보존을 동시에 확인한다. 높은 분류 F1이나 100% no-op 보존만으로 성공이라고 쓰지 않는다. 1-pass라는 명칭 자체의 최초성은 주장하지 않는다.

**원본 근거:**
- E14: https://drive.google.com/file/d/1D68pg1E_N3NJBZinjNB2XiyKA62ZuPM8/view
- E16 심사대응 원형: https://drive.google.com/file/d/1uNkcqWLLYX6JbCheqdkphFaZRl6HPdjB/view
- E17: https://drive.google.com/file/d/1EQ_y8fGHL4CT5bAdvqafEOwuc3oh668o/view
- E19: https://drive.google.com/file/d/1W4s6Qkeg7UTm32KlfwRi44XoII-8OBcq/view

E15의 utility 관점은 참고했지만, 기존 고차원 hidden utility predictor를 새 4-state 라우터의 기여로 바꿔 부르지 않는다. E20의 A/B 첫 토큰 통계와 Packed-dependent 방향은 사용하지 않는다.